# N142 · 折半搜索与指数优化

**目标：** 把全集枚举拆成两侧并正确合并。

**先修：** N125, N042, N101。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 半集子集和；排序/二分；按元素个数分桶；输出规模；时间空间代价。

## 从问题到算法

当n约40时，完整2^n枚举可能太大，而2^(n/2)仍可存储。折半法枚举左右半部，随后把两半匹配问题交给排序与二分。若要求两组大小相同，只按和排序不够，还要按选中元素个数分桶，确保左右数量相加为n/2。

## 最小实现

**本章接口：** `subset_sums(nums)`、`min_abs_difference(nums, goal)`、`minimum_difference_equal_size(nums)`

In [1]:
from bisect import bisect_left

def subset_sums(nums):
    sums=[0]
    for x in nums: sums += [s+x for s in sums]
    return sums

def min_abs_difference(nums,goal):
    mid=len(nums)//2; left=subset_sums(nums[:mid]); right=sorted(subset_sums(nums[mid:])); answer=abs(goal)
    for a in left:
        p=bisect_left(right,goal-a)
        for j in (p-1,p):
            if 0<=j<len(right): answer=min(answer,abs(a+right[j]-goal))
        if answer==0: return 0
    return answer

def _sums_by_size(nums):
    buckets=[[] for _ in range(len(nums)+1)]; buckets[0]=[0]
    for i,x in enumerate(nums):
        for count in range(i+1,0,-1): buckets[count].extend(s+x for s in buckets[count-1])
    return buckets

def minimum_difference_equal_size(nums):
    if len(nums)%2: raise ValueError('even input length required')
    half=len(nums)//2; left=_sums_by_size(nums[:half]); right=_sums_by_size(nums[half:]); total=sum(nums); answer=float('inf')
    for count,values in enumerate(left):
        candidates=sorted(right[half-count])
        for a in values:
            target=(total-2*a+1)//2; p=bisect_left(candidates,target)
            for j in (p-1,p):
                if 0<=j<len(candidates): answer=min(answer,abs(total-2*(a+candidates[j])))
    return answer

## 正确性、前提与复杂度

每个完整子集唯一分解为左右两个子集；二分目标的前驱或后继必有距离目标最近的右和。等大小问题只组合count与half−count桶，精确排除数量不合法的方案。比较目标使用整数二倍表达式，不把巨大整数转换为浮点。

**代价：** 折半后时间通常O(n·2^(n/2))（含排序/二分），空间O(2^(n/2))；仍是指数算法。subset_sums保留不同子集导致的重复和，不能把它误当作不同和集合。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[3,9,7,3]; half=len(a)//2; left=_sums_by_size(a[:half]); right=_sums_by_size(a[half:])
show_table(['左侧选取数量','左和候选','对应右侧数量','右和候选'],[(k,left[k],half-k,right[half-k]) for k in range(half+1)])
print('等大小最小差',minimum_difference_equal_size(a))

左侧选取数量,左和候选,对应右侧数量,右和候选
0,[0],2,[10]
1,"[3, 9]",1,"[7, 3]"
2,[12],0,[0]


等大小最小差 2


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert min_abs_difference([5,-7,3,5],6)==0
assert minimum_difference_equal_size([3,9,7,3])==2
assert minimum_difference_equal_size([-36,36])==72
assert minimum_difference_equal_size([])==0
from random import Random
from itertools import combinations
rng=Random(142)
for n in range(11):
    for _ in range(20):
        a=[rng.randrange(-20,21) for _ in range(n)]; goal=rng.randrange(-30,31)
        sums=[sum(a[i] for i in range(n) if mask>>i&1) for mask in range(1<<n)]
        assert sorted(subset_sums(a))==sorted(sums)
        assert min_abs_difference(a,goal)==min(abs(x-goal) for x in sums)
        if n%2==0:
            want=min(abs(sum(a)-2*sum(a[i] for i in ids)) for ids in combinations(range(n),n//2))
            assert minimum_difference_equal_size(a)==want
assert minimum_difference_equal_size([10**30,10**30+1])==1
print('N142: 所有本章断言通过')

N142: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 等大小划分时按选取数量分桶。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 比较2^n和2^(n/2)的实际状态数。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 1755. Closest Subsequence Sum（canonical）
- 2035. Partition Array Into Two Arrays to Minimize Sum Difference（canonical）
- 805. Split Array With Same Average（extension）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。